# 02 — ConvNeXt-Tiny Baseline and Controlled Training Experiments (B0–B4)

**Research question.** Do conventional augmentation, class weighting, grade-3 sampling, or train-only label-conflict selection improve frozen-validation DR grading? Ten recorded 20-epoch runs are exploratory validation evidence, **not final test**. [Evidence JSON](../novelty/convnext_b0_b4/validation_evidence.json) stores all 200 original epoch records and figure hashes. [Readiness protocol](../docs/FOUNDATION_TRAINING_READINESS.md) records controls and limitations.

The executable source in Git supports B0–B2. Original B3/B4 runners and selection artifacts remain external; those runs are **analyzable** here but cannot be claimed executable from this checkout alone.

## 1. Datasets, license, and design

APTOS2019 is supplied via [Kaggle](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data), DDR via [OIA-DDR](https://github.com/nkicsl/DDR-dataset). No images are embedded unless redistributable. Frozen train/validation manifests use seed 42 and separate datasets. B4-Q/R are **APTOS only**, since audited DDR manifests had no identical-SHA conflicting-label groups. APTOS B4-Q removes 50 rows in 24 conflicting groups; B4-R removes 50 class-matched rows in 27 nonconflicting whole-SHA groups. Neither uses validation performance for row selection.

All completed runs use 20 epochs. B0 is vanilla; B1 applies conservative training-only augmentation; B2 uses weighted CE; B3 adds grade-3 sampling to B1; B4-Q/R are train-only selection experiments. B3/B4 implementation cannot be substituted with current B0–B2 code.

## 2. Environment and execution controls

This setup checks the repository and external locations without download, install or inference. On Colab, clone the repository first and set `DR_RESEARCH_ROOT`, `DR_ARTIFACT_ROOT`, `DR_OUTPUT_ROOT`; provide authorized datasets and pinned official ConvNeXt snapshot separately. On HPC, use the audited environment and external paths. Rerunning is safe.

In [ ]:
from pathlib import Path
import os, sys, importlib.util, hashlib, json, csv
from IPython.display import display, Markdown

def show_table(rows, columns):
    if not rows:
        print("No recorded rows available")
        return
    header = "| " + " | ".join(columns) + " |"
    rule = "| " + " | ".join("---" for _ in columns) + " |"
    values = []
    for row in rows:
        values.append("| " + " | ".join(f"{row.get(k, '—'):.4f}" if isinstance(row.get(k), float)
                                      else str(row.get(k, '—')) for k in columns) + " |")
    display(Markdown("\n".join([header, rule, *values])))

def find_root():
    configured = os.environ.get("DR_RESEARCH_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "README.md").is_file():
            raise FileNotFoundError(f"DR_RESEARCH_ROOT is not a repository: {root}")
        return root
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / "README.md").is_file() and (root / "novelty_integrated" / "e5").is_dir():
            return root.resolve()
    raise FileNotFoundError("Set DR_RESEARCH_ROOT to the cloned repository")

ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ.get("DR_ARTIFACT_ROOT", ROOT.parent / "Kel_13")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("DR_OUTPUT_ROOT", ROOT.parent / "research_new_runs")).expanduser().resolve()

def verified_example(dataset, image_path, split):
    """Verify a display image is exactly a frozen train/validation manifest row."""
    if split not in {"train", "val"}:
        raise ValueError("Only train/val examples are permitted")
    manifest = ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv"
    if not manifest.is_file() or not manifest.with_suffix(".json").is_file():
        return False
    if hashlib.sha256(manifest.read_bytes()).hexdigest() != MANIFEST_SHA256[dataset]:
        raise ValueError("Frozen manifest hash mismatch")
    target = Path(image_path).expanduser().resolve()
    root = (ARTIFACT_ROOT / "datasets" / dataset).resolve()
    with manifest.open(encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            if row["split"] == "test":
                continue  # do not inspect test path, ID, label or hash
            if row["split"] != split:
                continue
            candidate = (root / row["image_path"]).resolve()
            if root not in candidate.parents:
                raise ValueError("Manifest path escapes dataset root")
            if candidate == target:
                if hashlib.sha256(target.read_bytes()).hexdigest() != row["sha256"]:
                    raise ValueError("Example image hash mismatch")
                print("Verified frozen", dataset, split, "example ID:", row["image_id"])
                return row["image_id"]
    raise ValueError("Example is not a frozen train/validation row; test access refused")
RUN_PREFLIGHT = False
RUN_MODEL_SANITY = False
RUN_SMOKE = False
RUN_TRAINING = False
RUN_RESUME = False
print("repository:", ROOT)
print("external artifacts:", ARTIFACT_ROOT, "available:", ARTIFACT_ROOT.is_dir())
print("external output (no directory created):", OUTPUT_ROOT)
print("Python:", sys.version.split()[0])
print({name: importlib.util.find_spec(name) is not None for name in ("torch", "torchvision", "transformers", "PIL", "cv2", "numpy", "pandas", "matplotlib")})
if importlib.util.find_spec("torch"):
    import torch
    print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
else:
    print("SKIPPED_MISSING_DATA: PyTorch-dependent model cells require the pinned project environment")

### Optional Drive mount and manifest identity

Drive mounting is opt-in. The second cell checks known frozen manifest SHA values if present; source preflight must also verify image hashes and group separation before training.

In [ ]:
# Run only when using Colab and after obtaining authorized dataset access.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Set DR_ARTIFACT_ROOT and DR_OUTPUT_ROOT, then rerun the setup cell")
else:
    print("Drive mount skipped; no packages or weights downloaded")

The following manifest check reads CSV bytes and sidecar presence only. It does not inspect test images or create any output.

In [ ]:
from pathlib import Path
MANIFEST_SHA256 = {"APTOS2019": "73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854",
                   "DDR": "7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1"}
for dataset, expected in MANIFEST_SHA256.items():
    csv_path = ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv"
    sidecar = csv_path.with_suffix(".json")
    if not csv_path.is_file() or not sidecar.is_file():
        print(dataset, "SKIPPED_MISSING_DATA: frozen CSV/sidecar not present on this host")
        continue
    actual = hashlib.sha256(csv_path.read_bytes()).hexdigest()
    assert actual == expected, f"{dataset} manifest SHA mismatch"
    print(dataset, "frozen CSV SHA-256 verified; sidecar present:", sidecar)
    print("For training, use the source runner's full image/group/hash preflight")

## 3. Source evidence and class distribution

The next cell verifies ten run identifiers, 20 original history objects each, and reports train/validation class support from the committed run configuration. The bar plot reflects **frozen-manifest metadata**, not synthetic images. This is safe to rerun.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("SKIPPED_MISSING_DATA: matplotlib unavailable; committed figures remain visible")
EVIDENCE_DIR = ROOT / "novelty/convnext_b0_b4"
evidence = json.loads((EVIDENCE_DIR / "validation_evidence.json").read_text(encoding="utf-8"))
runs = evidence["runs"]
assert len(runs) == 10 and sum(len(r["history"]) for r in runs) == 200
assert all([h["epoch"] for h in r["history"]] == list(range(1, 21)) for r in runs)
print("Verified 10 runs / 200 recorded epochs:", [r["id"] for r in runs])
for dataset in ("APTOS2019", "DDR"):
    baseline = next(r for r in runs if r["dataset"] == dataset and r["arm"] == "B0")
    support = baseline["configuration"]["manifest"]["grade_support"]
    print(dataset, "train/validation support:", support)
    if plt is not None:
        x = list(range(5)); plt.figure(figsize=(6, 3))
        plt.bar([v-.18 for v in x], support["train"], width=.36, label="train")
        plt.bar([v+.18 for v in x], support["val"], width=.36, label="validation")
        plt.xticks(x); plt.xlabel("DR grade"); plt.ylabel("Images")
        plt.title(dataset + " frozen split class support"); plt.legend(); plt.show()

## 4. Real images, preprocessing and augmentation

The original runner uses the pinned official `facebook/convnext-tiny-224` image processor at 224 px. B1's training-only augmentation is horizontal flip p=.50, rotation p=.50 within ±7°, brightness p=.30 at .90–1.10 and contrast p=.30 at .90–1.10; validation stays deterministic. The source order is augmentation **before** the official processor. The next cell shows a real train/validation image only when the user supplies an authorized local path and split. It does not write image data or open a test image. Missing data yields `SKIPPED_MISSING_DATA`.

In [ ]:
sample = os.environ.get("DR_SAMPLE_IMAGE")
split = os.environ.get("DR_SAMPLE_SPLIT", "")
SAMPLE_VERIFIED = False
SAMPLE_ROW_ID = None
if not sample:
    print("SKIPPED_MISSING_DATA: set DR_SAMPLE_IMAGE and DR_SAMPLE_SPLIT=train or val")
elif split not in {"train", "val"}:
    print("SKIPPED_MISSING_DATA: test sample display is disabled")
elif not (SAMPLE_ROW_ID := verified_example(os.environ.get("DR_SAMPLE_DATASET", "APTOS2019"), sample, split)):
    print("SKIPPED_MISSING_DATA: frozen manifest and sidecar unavailable for image verification")
else:
    SAMPLE_VERIFIED = True
    from PIL import Image, ImageOps
    with Image.open(sample) as im:
        raw = ImageOps.exif_transpose(im).convert("RGB")
        display(raw.resize((224, 224)))
    print("For exact B1 stochastic augmentation, use novelty.foundation_adaptation inspect-augmentation with frozen train inputs")

### Verified official processor before/after and train-only B1 example

When `DR_SAMPLE_IMAGE` points to a manifest-verified train/validation sample and `DR_CONVNEXT_SNAPSHOT` points to the pinned official snapshot, this cell audits the weight/config hashes, loads **only the processor**, and displays the real processor input/output. For a training sample, it uses the verified manifest image ID to call the actual B1 augmentation at seed 42/epoch 1 before processing. It never runs a model forward pass or writes a figure. Missing inputs are explicitly skipped.

In [ ]:
snapshot_dir = os.environ.get("DR_CONVNEXT_SNAPSHOT", "")
if not SAMPLE_VERIFIED or not snapshot_dir:
    print("SKIPPED_MISSING_DATA: train/val sample or pinned official snapshot unavailable")
else:
    from novelty import foundation_adaptation as fa
    from PIL import Image, ImageOps
    import numpy as np
    expected = next(r for r in runs if r["dataset"] == os.environ.get("DR_SAMPLE_DATASET", "APTOS2019")
                    and r["arm"] == "B1")["configuration"]["snapshot"]
    audited = fa.audit_snapshot("convnext_tiny", snapshot_dir, expected["revision"], expected["weight_sha256"])
    assert audited["processor_config_sha256"] == expected["processor_config_sha256"]
    processor = fa.load_local_processor(audited)
    with Image.open(sample) as im:
        rgb = ImageOps.exif_transpose(im).convert("RGB")
    image_id = SAMPLE_ROW_ID
    if split == "train" and image_id:
        from novelty import foundation_augmentation
        augmented, _ = foundation_augmentation.apply(rgb, seed=42, epoch=1, image_id=image_id)
        display(rgb.resize((224, 224)), augmented.resize((224, 224)))
        processed = processor(images=augmented, return_tensors="pt")["pixel_values"]
    else:
        print("No train image ID: deterministic official processing only")
        processed = processor(images=rgb, return_tensors="pt")["pixel_values"]
    mean = np.asarray(processor.image_mean)[:, None, None]
    std = np.asarray(processor.image_std)[:, None, None]
    array = np.clip(processed[0].numpy() * std + mean, 0, 1).transpose(1, 2, 0)
    display(Image.fromarray((array * 255).round().astype(np.uint8)))
    print("official tensor:", tuple(processed.shape), processed.dtype)

The source `inspect-augmentation` command creates a real train-only gallery outside Git. The cell below prints its audited CLI; it is **disabled** by default because it reads images and writes an external report. It must use the exact pinned snapshot and training manifest. No gallery is fabricated.

In [ ]:
print("python -m novelty.foundation_adaptation inspect-augmentation --dataset APTOS2019 "
      "--artifact-root <KEL_13> --model-kind convnext_tiny --model-dir <PINNED_SNAPSHOT> "
      "--revision <REVISION> --weight-sha256 <SHA256> --output-dir <EXTERNAL_OUTPUT>")
print("SKIPPED: not executed in this notebook")

## 5. Model, loss and optimization

`novelty.foundation_adaptation` loads the official pinned ConvNeXt-Tiny backbone and processor, attaches a five-class linear head, and full fine-tunes with AdamW. The actual run metadata records batch, accumulation, BF16, backbone/head learning rates and weight decay. B2 uses training-only class counts `n_c` and `w_c=n_c^(-1/2)/(Σ_j n_j n_j^(-1/2)/N_train)` in weighted CE; validation uses ordinary CE. B3/B4 historical executable source is external. The next cell prints the **recorded** architecture/protocol fields without duplicating the model definition. Actual parameter counting of a fresh model is optional and requires verifying the external official snapshot.

The source-defined inference path is shared across arms; intervention occurs in training data or objective, never in the final five-class prediction head.

```mermaid
flowchart LR
    I[Authorized RGB train/validation image] --> P[Official 224px processor]
    P --> B[Pinned ConvNeXt-Tiny backbone]
    B --> H[Linear 768→5 head]
    H --> G[Five-grade logits and predictions]
```

In [ ]:
for dataset in ("APTOS2019", "DDR"):
    b0 = next(r for r in runs if r["dataset"] == dataset and r["arm"] == "B0")
    c = b0["configuration"]
    print(dataset, {k: c.get(k) for k in ("model_kind", "mode", "epochs", "seed", "micro_batch", "effective_batch", "optimizer", "backbone_lr", "head_lr", "weight_decay", "precision", "selection")})
print("Exact parameter count requires constructing the pinned model; see source smoke report when snapshot exists")

The following **optional** CPU-only cell checks the pinned official snapshot hashes, loads the original ConvNeXt model and five-class linear head, then counts parameters from actual tensors. It does not run forward inference or download weights. Keep `RUN_MODEL_SANITY=False` if the snapshot is absent or RAM is limited.

In [ ]:
if not RUN_MODEL_SANITY:
    print("SKIPPED: actual model construction disabled")
else:
    from novelty import foundation_adaptation as fa
    import torch
    reference = next(r for r in runs if r["dataset"] == "APTOS2019" and r["arm"] == "B1")["configuration"]["snapshot"]
    snapshot_dir = os.environ.get("DR_CONVNEXT_SNAPSHOT", "")
    if not snapshot_dir:
        print("SKIPPED_MISSING_DATA: pinned official snapshot unavailable")
    else:
        audited = fa.audit_snapshot("convnext_tiny", snapshot_dir, reference["revision"], reference["weight_sha256"])
        assert audited["config_sha256"] == reference["config_sha256"]
        assert audited["processor_config_sha256"] == reference["processor_config_sha256"]
        backbone, _ = fa.load_local_backbone(audited, trust_legacy_bin=True)
        head = torch.nn.Linear(768, 5)
        print("Actual constructed parameters:", sum(p.numel() for p in backbone.parameters()) +
              sum(p.numel() for p in head.parameters()))

### Controlled new-run guard: B0–B2 only

The source CLI can run `preflight`, `smoke`, then `train` for B0/B1/B2. It enforces pinned snapshot SHA, manifest and image integrity, checkpoint selection by maximum validation macro-F1 (then lower validation loss, then earlier epoch), complete run metadata, and guarded resume. The next cell constructs the actual `preflight` command; all external paths and hash values must be supplied and checked first. It performs no action unless `RUN_PREFLIGHT=True`. It is not a historical B3/B4 rerun.

In [ ]:
import subprocess
MODEL_DIR = os.environ.get("DR_CONVNEXT_SNAPSHOT", "")
MODEL_REVISION = os.environ.get("DR_CONVNEXT_REVISION", "")
MODEL_WEIGHT_SHA256 = os.environ.get("DR_CONVNEXT_WEIGHT_SHA256", "")
dataset = "APTOS2019"
arm = "B1"  # only B0, B1, B2 supported by the tracked source
base = [sys.executable, "-m", "novelty.foundation_adaptation", "preflight",
        "--dataset", dataset, "--artifact-root", str(ARTIFACT_ROOT),
        "--model-kind", "convnext_tiny", "--model-dir", MODEL_DIR,
        "--revision", MODEL_REVISION, "--weight-sha256", MODEL_WEIGHT_SHA256]
print("Source preflight:", base)
if RUN_PREFLIGHT:
    if not all((MODEL_DIR, MODEL_REVISION, MODEL_WEIGHT_SHA256)) or not ARTIFACT_ROOT.is_dir():
        print("SKIPPED_MISSING_DATA: pinned snapshot/hash or frozen artifacts unavailable")
    else:
        subprocess.run(base, cwd=ROOT, check=True)
else:
    print("SKIPPED: RUN_PREFLIGHT=False")

### Optional source smoke check

The next cell calls the tracked B0–B2 source's `smoke` command only when `RUN_SMOKE=True`, after preflight and snapshot/hash configuration. It writes a **separate fresh external smoke directory** and produces the report required by training. It never points to an historical output. This is a small training-step check, so leave it disabled until GPU resources and permissions are available.

In [ ]:
SMOKE_DIR = OUTPUT_ROOT / dataset / f"convnext_{arm}_smoke_new"
SMOKE_REPORT = os.environ.get("DR_SMOKE_REPORT", str(SMOKE_DIR / "smoke.json"))
smoke_args = [sys.executable, "-m", "novelty.foundation_adaptation", "smoke",
    "--dataset", dataset, "--artifact-root", str(ARTIFACT_ROOT),
    "--model-kind", "convnext_tiny", "--model-dir", MODEL_DIR,
    "--revision", MODEL_REVISION, "--weight-sha256", MODEL_WEIGHT_SHA256,
    "--experiment", arm, "--mode", "full", "--batch-size", "32",
    "--precision", "bf16", "--output-dir", str(SMOKE_DIR)]
print("Source smoke:", smoke_args)
if RUN_SMOKE and RUN_PREFLIGHT:
    if SMOKE_DIR.exists() or ROOT in SMOKE_DIR.parents or not all((MODEL_DIR, MODEL_REVISION, MODEL_WEIGHT_SHA256)):
        raise RuntimeError("Use fresh external smoke directory and verified snapshot")
    subprocess.run(smoke_args, cwd=ROOT, check=True)
else:
    print("SKIPPED: RUN_SMOKE=False")

The source `smoke` command validates a tiny batch and records resource use; the source `train` command owns optimizer, epoch metrics, checkpoint hashes, resume and learning curves. This training cell is disabled and requires explicit report paths plus preflight. Use a **new external output directory**; do not point it at any B0–B4 historical run. It cannot invoke B3/B4 from this repository.

In [ ]:
new_run = OUTPUT_ROOT / dataset / f"convnext_{arm}_new"
train_args = [sys.executable, "-m", "novelty.foundation_adaptation", "train",
    "--dataset", dataset, "--artifact-root", str(ARTIFACT_ROOT),
    "--model-kind", "convnext_tiny", "--model-dir", MODEL_DIR,
    "--revision", MODEL_REVISION, "--weight-sha256", MODEL_WEIGHT_SHA256,
    "--experiment", arm, "--mode", "full", "--smoke-report", SMOKE_REPORT,
    "--output-dir", str(new_run), "--epochs", "20", "--micro-batch", "32",
    "--accumulation", "1", "--precision", "bf16"]
print("New B0/B1/B2 training command:", train_args)
if RUN_TRAINING and RUN_PREFLIGHT and RUN_SMOKE:
    if arm not in {"B0", "B1", "B2"} or not SMOKE_REPORT or not MODEL_DIR or ROOT in new_run.parents:
        raise RuntimeError("Missing frozen prerequisites or output path is inside Git")
    subprocess.run(train_args, cwd=ROOT, check=True)
else:
    print("SKIPPED: RUN_TRAINING=False / required guards not enabled")

## 6. Validation comparison: all completed runs

The next cell reads only `best.validation` from the ten verified records. It displays accuracy, macro-F1, weighted-F1, QWK, MAE, and the validation-selected epoch. No test result or recalculated prediction is introduced. Re-running is safe.

In [ ]:
comparison = []
for r in runs:
    v = r["best"]["validation"]
    comparison.append({"dataset": r["dataset"], "arm": r["arm"], "best_epoch": r["best"]["epoch"],
        "accuracy": v["accuracy"], "macro_f1": v["macro_f1"], "weighted_f1": v["weighted_f1"],
        "qwk": v["quadratic_weighted_kappa"], "grade_mae": v["grade_mae"]})
show_table(comparison, ["dataset", "arm", "best_epoch", "accuracy", "macro_f1", "weighted_f1", "qwk", "grade_mae"])

**Committed validation comparison (best epoch; generated directly from `validation_evidence.json`).**

| Dataset | Arm | Epoch | Accuracy | Macro-F1 | Weighted-F1 | QWK | Grade MAE |
| --- | --- | --- | --- | --- | --- | --- | --- |
| APTOS2019 | B0 | 17 | 0.814208 | 0.673686 | 0.817133 | 0.889230 | 0.243169 |
| APTOS2019 | B1 | 16 | 0.830601 | 0.701124 | 0.832144 | 0.898623 | 0.221311 |
| APTOS2019 | B2 | 7 | 0.797814 | 0.659987 | 0.802774 | 0.882524 | 0.262295 |
| APTOS2019 | B3 | 20 | 0.827869 | 0.700624 | 0.829804 | 0.899896 | 0.221311 |
| APTOS2019 | B4-Q | 11 | 0.819672 | 0.688319 | 0.823274 | 0.893995 | 0.234973 |
| APTOS2019 | B4-R | 9 | 0.827869 | 0.707897 | 0.831054 | 0.898937 | 0.221311 |
| DDR | B0 | 8 | 0.807508 | 0.619817 | 0.795273 | 0.820307 | 0.312300 |
| DDR | B1 | 11 | 0.826677 | 0.652146 | 0.818245 | 0.843538 | 0.275559 |
| DDR | B2 | 5 | 0.770767 | 0.623084 | 0.774761 | 0.800902 | 0.361022 |
| DDR | B3 | 11 | 0.818690 | 0.651006 | 0.813732 | 0.843194 | 0.281949 |

### Per-grade precision, recall, F1 and support

Macro-F1 weights every grade equally, but it does **not** guarantee higher recall for grade 1 or grade 3. This table exposes each grade's support and confusion pattern. The code reads recorded best-epoch metrics, not a new inference pass.

In [ ]:
grade_rows = []
for r in runs:
    for g in r["best"]["validation"]["per_grade"]:
        grade_rows.append({"dataset": r["dataset"], "arm": r["arm"], "grade": g["grade"],
             "support": g["support"], "precision": g["precision"], "recall": g["recall"],
             "f1": g["f1"], "correct": g["correct"]})
show_table(grade_rows, ["dataset", "arm", "grade", "support", "precision", "recall", "f1", "correct"])
show_table([g for g in grade_rows if g["grade"] in (1, 3)],
           ["dataset", "arm", "grade", "support", "recall", "f1"])

**Per-grade best-validation evidence (all 50 class rows; generated from the same JSON).**

| Dataset | Arm | Grade | Support | Correct | Precision | Recall | F1 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| APTOS2019 | B0 | 0 | 180 | 174 | 0.994286 | 0.966667 | 0.980282 |
| APTOS2019 | B0 | 1 | 35 | 23 | 0.500000 | 0.657143 | 0.567901 |
| APTOS2019 | B0 | 2 | 101 | 76 | 0.760000 | 0.752475 | 0.756219 |
| APTOS2019 | B0 | 3 | 20 | 10 | 0.625000 | 0.500000 | 0.555556 |
| APTOS2019 | B0 | 4 | 30 | 15 | 0.517241 | 0.500000 | 0.508475 |
| APTOS2019 | B1 | 0 | 180 | 174 | 0.994286 | 0.966667 | 0.980282 |
| APTOS2019 | B1 | 1 | 35 | 25 | 0.543478 | 0.714286 | 0.617284 |
| APTOS2019 | B1 | 2 | 101 | 78 | 0.764706 | 0.772277 | 0.768473 |
| APTOS2019 | B1 | 3 | 20 | 9 | 0.642857 | 0.450000 | 0.529412 |
| APTOS2019 | B1 | 4 | 30 | 18 | 0.620690 | 0.600000 | 0.610169 |
| APTOS2019 | B2 | 0 | 180 | 171 | 0.988439 | 0.950000 | 0.968839 |
| APTOS2019 | B2 | 1 | 35 | 21 | 0.456522 | 0.600000 | 0.518519 |
| APTOS2019 | B2 | 2 | 101 | 74 | 0.747475 | 0.732673 | 0.740000 |
| APTOS2019 | B2 | 3 | 20 | 11 | 0.578947 | 0.550000 | 0.564103 |
| APTOS2019 | B2 | 4 | 30 | 15 | 0.517241 | 0.500000 | 0.508475 |
| APTOS2019 | B3 | 0 | 180 | 174 | 0.994286 | 0.966667 | 0.980282 |
| APTOS2019 | B3 | 1 | 35 | 23 | 0.534884 | 0.657143 | 0.589744 |
| APTOS2019 | B3 | 2 | 101 | 78 | 0.750000 | 0.772277 | 0.760976 |
| APTOS2019 | B3 | 3 | 20 | 10 | 0.588235 | 0.500000 | 0.540541 |
| APTOS2019 | B3 | 4 | 30 | 18 | 0.666667 | 0.600000 | 0.631579 |
| APTOS2019 | B4-Q | 0 | 180 | 175 | 0.994318 | 0.972222 | 0.983146 |
| APTOS2019 | B4-Q | 1 | 35 | 23 | 0.534884 | 0.657143 | 0.589744 |
| APTOS2019 | B4-Q | 2 | 101 | 73 | 0.768421 | 0.722772 | 0.744898 |
| APTOS2019 | B4-Q | 3 | 20 | 11 | 0.500000 | 0.550000 | 0.523810 |
| APTOS2019 | B4-Q | 4 | 30 | 18 | 0.600000 | 0.600000 | 0.600000 |
| APTOS2019 | B4-R | 0 | 180 | 174 | 0.994286 | 0.966667 | 0.980282 |
| APTOS2019 | B4-R | 1 | 35 | 21 | 0.512195 | 0.600000 | 0.552632 |
| APTOS2019 | B4-R | 2 | 101 | 77 | 0.770000 | 0.762376 | 0.766169 |
| APTOS2019 | B4-R | 3 | 20 | 14 | 0.560000 | 0.700000 | 0.622222 |
| APTOS2019 | B4-R | 4 | 30 | 17 | 0.680000 | 0.566667 | 0.618182 |
| DDR | B0 | 0 | 626 | 568 | 0.877898 | 0.907348 | 0.892380 |
| DDR | B0 | 1 | 63 | 5 | 0.263158 | 0.079365 | 0.121951 |
| DDR | B0 | 2 | 448 | 360 | 0.753138 | 0.803571 | 0.777538 |
| DDR | B0 | 3 | 24 | 15 | 0.517241 | 0.625000 | 0.566038 |
| DDR | B0 | 4 | 91 | 63 | 0.797468 | 0.692308 | 0.741176 |
| DDR | B1 | 0 | 626 | 574 | 0.889922 | 0.916933 | 0.903226 |
| DDR | B1 | 1 | 63 | 11 | 0.314286 | 0.174603 | 0.224490 |
| DDR | B1 | 2 | 448 | 371 | 0.781053 | 0.828125 | 0.803900 |
| DDR | B1 | 3 | 24 | 10 | 0.769231 | 0.416667 | 0.540541 |
| DDR | B1 | 4 | 91 | 69 | 0.821429 | 0.758242 | 0.788571 |
| DDR | B2 | 0 | 626 | 522 | 0.893836 | 0.833866 | 0.862810 |
| DDR | B2 | 1 | 63 | 16 | 0.235294 | 0.253968 | 0.244275 |
| DDR | B2 | 2 | 448 | 341 | 0.728632 | 0.761161 | 0.744541 |
| DDR | B2 | 3 | 24 | 15 | 0.428571 | 0.625000 | 0.508475 |
| DDR | B2 | 4 | 91 | 71 | 0.731959 | 0.780220 | 0.755319 |
| DDR | B3 | 0 | 626 | 579 | 0.881279 | 0.924920 | 0.902572 |
| DDR | B3 | 1 | 63 | 13 | 0.270833 | 0.206349 | 0.234234 |
| DDR | B3 | 2 | 448 | 347 | 0.801386 | 0.774554 | 0.787741 |
| DDR | B3 | 3 | 24 | 12 | 0.545455 | 0.500000 | 0.521739 |
| DDR | B3 | 4 | 91 | 74 | 0.804348 | 0.813187 | 0.808743 |

### Original 200 epoch records and learning curves

The history objects remain intact in `validation_evidence.json`, including every field present in each source history. The following code plots recorded training loss and validation macro-F1; missing training metrics remain missing rather than inferred. The existing SVG curves are also embedded below for GitHub viewing without a kernel.

In [ ]:
for dataset in ("APTOS2019", "DDR"):
    subset = [r for r in runs if r["dataset"] == dataset]
    if plt is None:
        print(dataset, "dynamic plot skipped; original SVG figures appear below")
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for r in subset:
        epoch = [h["epoch"] for h in r["history"]]
        axes[0].plot(epoch, [h["train"].get("loss") for h in r["history"]], label=r["arm"])
        axes[1].plot(epoch, [h["validation"]["macro_f1"] for h in r["history"]], label=r["arm"])
    axes[0].set(title=dataset+" train loss", xlabel="epoch", ylabel="loss")
    axes[1].set(title=dataset+" validation macro-F1", xlabel="epoch", ylabel="macro-F1")
    for ax in axes: ax.legend()
    plt.tight_layout(); plt.show()

#### APTOS2019: recorded curves and confusion matrices

Each curve is linked to the corresponding best-epoch validation matrix (rows actual, columns predicted). Compare minority-grade **recall and support** alongside macro-F1; these plots use only existing committed evidence.

**APTOS2019 B0 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![APTOS2019 B0 training curves](../novelty/convnext_b0_b4/APTOS2019_B0_learning_curves.svg)

![APTOS2019 B0 best-validation confusion matrix](../novelty/convnext_b0_b4/APTOS2019_B0_confusion_matrix.png)

**APTOS2019 B1 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![APTOS2019 B1 training curves](../novelty/convnext_b0_b4/APTOS2019_B1_learning_curves.svg)

![APTOS2019 B1 best-validation confusion matrix](../novelty/convnext_b0_b4/APTOS2019_B1_confusion_matrix.png)

**APTOS2019 B2 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![APTOS2019 B2 training curves](../novelty/convnext_b0_b4/APTOS2019_B2_learning_curves.svg)

![APTOS2019 B2 best-validation confusion matrix](../novelty/convnext_b0_b4/APTOS2019_B2_confusion_matrix.png)

**APTOS2019 B3 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![APTOS2019 B3 training curves](../novelty/convnext_b0_b4/APTOS2019_B3_learning_curves.svg)

![APTOS2019 B3 best-validation confusion matrix](../novelty/convnext_b0_b4/APTOS2019_B3_confusion_matrix.png)

**APTOS2019 B4_Q — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![APTOS2019 B4_Q training curves](../novelty/convnext_b0_b4/APTOS2019_B4_Q_learning_curves.svg)

![APTOS2019 B4_Q best-validation confusion matrix](../novelty/convnext_b0_b4/APTOS2019_B4_Q_confusion_matrix.png)

**APTOS2019 B4_R — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![APTOS2019 B4_R training curves](../novelty/convnext_b0_b4/APTOS2019_B4_R_learning_curves.svg)

![APTOS2019 B4_R best-validation confusion matrix](../novelty/convnext_b0_b4/APTOS2019_B4_R_confusion_matrix.png)

#### DDR: recorded curves and confusion matrices

Each curve is linked to the corresponding best-epoch validation matrix (rows actual, columns predicted). Compare minority-grade **recall and support** alongside macro-F1; these plots use only existing committed evidence.

**DDR B0 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![DDR B0 training curves](../novelty/convnext_b0_b4/DDR_B0_learning_curves.svg)

![DDR B0 best-validation confusion matrix](../novelty/convnext_b0_b4/DDR_B0_confusion_matrix.png)

**DDR B1 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![DDR B1 training curves](../novelty/convnext_b0_b4/DDR_B1_learning_curves.svg)

![DDR B1 best-validation confusion matrix](../novelty/convnext_b0_b4/DDR_B1_confusion_matrix.png)

**DDR B2 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![DDR B2 training curves](../novelty/convnext_b0_b4/DDR_B2_learning_curves.svg)

![DDR B2 best-validation confusion matrix](../novelty/convnext_b0_b4/DDR_B2_confusion_matrix.png)

**DDR B3 — validation evidence.** Inspect training versus validation trends and grade-1/3 cells; this figure is descriptive, not an independently confirmed improvement.

![DDR B3 training curves](../novelty/convnext_b0_b4/DDR_B3_learning_curves.svg)

![DDR B3 best-validation confusion matrix](../novelty/convnext_b0_b4/DDR_B3_confusion_matrix.png)

## 7. Aggregate error analysis and limits

B1 improves APTOS validation macro-F1 over B0 (0.701124 versus 0.673686); B4-R reaches 0.707897, whereas B4-Q is 0.688319. DDR B1/B3 are close (0.652146/0.651006). These are single-seed, validation-selected results, so no significance or causal gain is established. Grade 1 and grade 3 have small support, and their recalls can move against the overall macro-F1. B3 oversampling repeats minority examples; B4 removal changes training size/content. Compare only matched protocols. No B0–B4 frozen final-test results are claimed here; 50-epoch extensions and CPU optimization are future plans.